In [10]:
# Imports / runtime setup (similar style to toy_experiments/test_GMM.ipynb)
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import functools
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from tqdm import trange

import distrax
import optax
from flax.training import train_state

from learning.module.target_examples.gmm40 import GMM40
from learning.module.target_examples.funnel import Funnel
from learning.module.gbs.gbs_loss import rnd_no_target, lv_loss_from_values
from learning.module.gbs.gbs_trainer import PISGRADNet


In [11]:
# Target distribution
dim = 10
# target = Funnel(dim=dim, sample_bounds=[-30.0, 30.0])
target = GMM40(dim=dim)

# Log-prob values for LV objective (no gradients needed through this)
target_log_prob = jax.jit(jax.vmap(target.log_prob))


In [12]:
# GBS-LV hyperparameters
seed = 0
batch_size = 512
num_steps = 25
iters = 500

init_std = 1.0
grad_clip = 1.0
lr = 1e-3


def noise_schedule(t: jnp.ndarray) -> jnp.ndarray:
    # Constant sigma schedule for a quick sanity test.
    return jnp.asarray(1.0, dtype=jnp.float32)


# Prior
prior = distrax.MultivariateNormalDiag(jnp.zeros(dim), jnp.ones(dim) * init_std)
prior_sampler = lambda key: jnp.squeeze(prior.sample(seed=key, sample_shape=(1,)))  # [D]
prior_log_prob = prior.log_prob


In [13]:
# Models + optimizer
key = jax.random.PRNGKey(seed)
key, k1, k2 = jax.random.split(key, 3)

model_cfg = dict(dim=dim, num_layers=2, num_hid=64)
fwd_model = PISGRADNet(**model_cfg)
bwd_model = PISGRADNet(**model_cfg)

fwd_params = fwd_model.init(
    k1,
    jnp.ones([batch_size, dim]),
    jnp.ones([batch_size, 1]),
    jnp.ones([batch_size, dim]),
)
bwd_params = bwd_model.init(
    k2,
    jnp.ones([batch_size, dim]),
    jnp.ones([batch_size, 1]),
    jnp.ones([batch_size, dim]),
)

optimizer = optax.chain(
    optax.zero_nans(),
    optax.clip(grad_clip),
    optax.adam(learning_rate=lr),
)
fwd_state = train_state.TrainState.create(apply_fn=fwd_model.apply, params=fwd_params, tx=optimizer)
bwd_state = train_state.TrainState.create(apply_fn=bwd_model.apply, params=bwd_params, tx=optimizer)


In [14]:
# Jitted sampler (no target inside) + jitted gradient step.
# NOTE: stop_grad must be static because rnd_no_target uses Python branching.
rnd_jit = jax.jit(
    rnd_no_target,
    static_argnums=(4, 5, 6, 7, 8),
)


def loss_wrapped(
    key,
    model_state,
    fwd_params,
    bwd_params,
    batch_size,
    prior_sampler,
    num_steps,
    noise_schedule,
    target_lp_vals,
):
    x0, xT, log_ratio = rnd_no_target(
        key,
        model_state,
        fwd_params,
        bwd_params,
        batch_size,
        prior_sampler,
        num_steps,
        noise_schedule,
        True,  # stop_grad
    )
    loss, aux, _ = lv_loss_from_values(x0, xT, log_ratio, prior_log_prob, target_lp_vals)
    return loss, aux


loss_grad = jax.jit(
    jax.grad(loss_wrapped, (2, 3), has_aux=True),
    static_argnums=(4, 5, 6, 7),
)


In [15]:
from matplotlib.colors import PowerNorm
# -------------------
# 2D targets (log-density)
# -------------------------
def target1_logprob(x):
    """Mixture of 2 Gaussians; x: [N,2] or [2]. returns logp: [N] or scalar."""
    x = jnp.atleast_2d(x)

    mean1 = jnp.array([1.0, 0.4])
    cov1 = 0.3 * jnp.array([[1.0, 0.3], [0.3, 1.0]])
    mean2 = jnp.array([-1.0, -0.4])
    cov2 = 0.1 * jnp.array([[1.0, -0.3], [-0.3, 1.0]])

    d1 = distrax.MultivariateNormalFullCovariance(mean1, cov1)
    d2 = distrax.MultivariateNormalFullCovariance(mean2, cov2)

    logw = jnp.log(jnp.array([0.4, 0.6]))
    lp = jnp.stack([d1.log_prob(x), d2.log_prob(x)], axis=-1)  # [N,2]
    out = jax.nn.logsumexp(lp + logw, axis=-1)                 # [N]
    return out.squeeze()


beta = -1.0
gamma = 0.45

def target2_logprob(z):
    """Energy-based; z: [N,2] or [2]. returns logp (unnormalized)."""
    z = jnp.atleast_2d(z)
    z1, z2 = z[:, 0:1], z[:, 1:2]

    r = jnp.hypot(z1, z2)

    logexp1 = -0.5 * jnp.square((z1 - 2.0) / 0.8)
    logexp2 = -0.5 * jnp.square((z1 + 2.0) / 0.8)
    log_mix = jax.nn.logsumexp(jnp.concatenate([logexp1, logexp2], axis=-1), axis=-1, keepdims=True)

    u = 0.5 * jnp.square((r - 4.0) / 0.4) - log_mix
    logp = beta * u
    return logp.squeeze()


def target3_logprob(z):
    """Petal/ring energy; expects z in [0,1]^2; returns logp (unnormalized)."""
    z = jnp.atleast_2d(z)
    X, Y = z[:, 0:1], z[:, 1:2]

    m = 3
    r0 = 0.65
    sr = 0.12

    x = 2.0 * (X - 0.5)
    y = 2.0 * (Y - 0.5)
    r = jnp.hypot(x, y)
    theta = jnp.arctan2(y, x)

    ring = jnp.exp(-0.5 * ((r - r0) / sr) ** 2)
    petals = jnp.cos(m * theta)
    U = jnp.tanh(1.6 * (ring * petals))  # (-1,1)
    logp = -beta * U
    return logp.squeeze()


# -------------------------
# Plot helper: target contour
# -------------------------
def plot_target_contour(ax, low, high, logprob_fn, n=200, levels=10, norm_gamma=0.45, title="target"):
    x, y = jnp.meshgrid(jnp.linspace(low[0], high[0], n),
                        jnp.linspace(low[1], high[1], n),
                        indexing="xy")
    grid = jnp.stack([x.reshape(-1), y.reshape(-1)], axis=-1)
    lp = logprob_fn(grid)
    Z = jnp.exp(jnp.clip(lp, a_min=-1000.0)).reshape(n, n)

    ctf = ax.contourf(np.array(x), np.array(y), np.array(Z),
                      levels=levels, cmap="viridis", norm=PowerNorm(norm_gamma))
    ax.set_title(title)
    ax.set_xlim(float(low[0]), float(high[0]))
    ax.set_ylim(float(low[1]), float(high[1]))
    ax.set_aspect("equal")
    return ctf


In [16]:
import jax
import jax.numpy as jnp
import numpy as np
import distrax
import optax
from flax.training import train_state
from tqdm import trange
import imageio

from learning.module.gbs.gbs_loss import (
    VP,
    Langevin,
    rnd_no_target,
    rnd_time_reversal_lv_no_target,
    lv_loss_from_values,
    lv_loss_from_rnd,
)
from learning.module.gbs.gbs_trainer import PISGRADNet


def run_gbs_toy(
    logprob_fn,
    low=jnp.array([0.0, 0.0]),
    high=jnp.array([1.0, 1.0]),
    dim=2,
    T=1000,
    batch_size=512,
    num_steps=25,
    lr=1e-3,
    init_std=0.5,
    snap_iters=None,
    gif_path=None,
    seed=0,
    loss_mode="tr_lv",  # "tr_lv" (sde_sampler LV) or "dis" (old kernel log-ratio)
):
    if snap_iters is None:
        snap_iters = []

    # Choose ONE process
    proc = VP(
        diff_coeff_sq_min=0.1,
        diff_coeff_sq_max=10.0,
        scale_diff_coeff=1.0,
        terminal_t=1.0,
        generative=False,
        sign=-1.0,
        include_base_drift=True,
    )
    # proc = Langevin(diff_coeff=1.0, terminal_t=1.0)

    key = jax.random.PRNGKey(seed)

    prior = distrax.MultivariateNormalDiag(
        loc=jnp.ones(dim) * 0.5,
        scale_diag=jnp.ones(dim) * init_std,
    )
    prior_sampler = lambda k: jnp.clip(
        jnp.squeeze(prior.sample(seed=k, sample_shape=(1,))), low, high
    )
    prior_log_prob = prior.log_prob

    model_cfg = dict(dim=dim, num_layers=2, num_hid=64)
    fwd_model = PISGRADNet(**model_cfg)
    bwd_model = PISGRADNet(**model_cfg)

    key, k1, k2 = jax.random.split(key, 3)
    fwd_params = fwd_model.init(
        k1, jnp.ones([batch_size, dim]), jnp.ones([batch_size, 1]), jnp.ones([batch_size, dim])
    )
    bwd_params = bwd_model.init(
        k2, jnp.ones([batch_size, dim]), jnp.ones([batch_size, 1]), jnp.ones([batch_size, dim])
    )

    opt = optax.chain(optax.zero_nans(), optax.clip(1.0), optax.adam(lr))
    fwd_state = train_state.TrainState.create(apply_fn=fwd_model.apply, params=fwd_params, tx=opt)
    bwd_state = train_state.TrainState.create(apply_fn=bwd_model.apply, params=bwd_params, tx=opt)

    # JIT samplers
    if loss_mode == "tr_lv":
        rnd_jit = jax.jit(rnd_time_reversal_lv_no_target, static_argnums=(3, 4, 5, 6, 7))
    elif loss_mode == "dis":
        rnd_jit = jax.jit(rnd_no_target, static_argnums=(4, 5, 6, 7, 8))
    else:
        raise ValueError(f"Unknown loss_mode: {loss_mode}")

    # JIT grad step
    if loss_mode == "tr_lv":
        def loss_wrapped(key, model_state, fwd_params, bwd_params):
            x0, xT, rnd_running = rnd_time_reversal_lv_no_target(
                key,
                model_state,
                fwd_params,
                batch_size,
                prior_sampler,
                num_steps,
                proc,
                True,
            )
            target_lp_vals = jnp.asarray(logprob_fn(xT)).reshape(-1)
            rnd_total = prior_log_prob(x0) + rnd_running - target_lp_vals
            loss, aux, _ = lv_loss_from_rnd(rnd_total, xT=xT)
            return loss, aux

        loss_grad = jax.jit(jax.grad(loss_wrapped, (2, 3), has_aux=True))
        hist = {k: [] for k in ["train/rnd_mean", "train/rnd_var", "train/xT_mean_norm"]}

    else:  # "dis"
        def loss_wrapped(key, model_state, fwd_params, bwd_params):
            x0, xT, log_ratio = rnd_no_target(
                key,
                model_state,
                fwd_params,
                bwd_params,
                batch_size,
                prior_sampler,
                num_steps,
                proc,   # you can also pass a callable sigma(step) here; proc works too
                True,
            )
            target_lp_vals = jnp.asarray(logprob_fn(xT)).reshape(-1)
            loss, aux = lv_loss_from_values(x0, xT, log_ratio, prior_log_prob, target_lp_vals)
            return loss, aux

        loss_grad = jax.jit(jax.grad(loss_wrapped, (2, 3), has_aux=True))
        hist = {k: [] for k in ["train/neg_elbo_mean", "train/neg_elbo_var",
                               "train/running_mean", "train/terminal_mean", "train/xT_mean_norm"]}

    frames = []
    last_xT = None

    for t in trange(T):
        key, k_step = jax.random.split(key)
        model_state = (fwd_state, bwd_state)

        if loss_mode == "tr_lv":
            x0, xT, _rnd_running = rnd_jit(
                k_step, model_state, fwd_state.params,
                batch_size, prior_sampler, num_steps, proc, True
            )
        else:
            x0, xT, _log_ratio = rnd_jit(
                k_step, model_state, fwd_state.params, bwd_state.params,
                batch_size, prior_sampler, num_steps, proc, True
            )

        last_xT = xT

        (fwd_grads, bwd_grads), aux = loss_grad(k_step, model_state, fwd_state.params, bwd_state.params)
        fwd_state = fwd_state.apply_gradients(grads=fwd_grads)
        bwd_state = bwd_state.apply_gradients(grads=bwd_grads)

        for k in hist:
            hist[k].append(float(aux[k]))

        if gif_path and (t in snap_iters):
            fig, ax = plt.subplots(1, 1, figsize=(5, 5))
            ctf = plot_target_contour(
                ax, low, high, logprob_fn, n=200, levels=10,
                norm_gamma=gamma, title=f"GBS xT (iter {t})"
            )
            fig.colorbar(ctf, ax=ax)
            pts = np.array(xT)
            ax.scatter(pts[:, 0], pts[:, 1], s=3, alpha=0.25, c="r")
            fig.tight_layout()
            fig.canvas.draw()
            frame = np.asarray(fig.canvas.buffer_rgba())[..., :3]
            frames.append(frame)
            plt.close(fig)

    if gif_path and frames:
        imageio.mimsave(gif_path, frames, fps=4)
        print("Saved GIF to:", gif_path)

    # Final big sample for saving/plotting
    key, k_final = jax.random.split(key)
    B = 2**14
    x0, xT_final, _ = rnd_time_reversal_lv_no_target(
        k_final, (fwd_state, bwd_state), fwd_state.params,
        B, prior_sampler, num_steps, proc, True
    )
    np.save("gbs_samples.npy", np.array(xT_final))

    plt.figure(figsize=(5, 5))
    plt.scatter(xT_final[:, 0], xT_final[:, 1], s=1, alpha=0.25)
    plt.xlim(float(low[0]), float(high[0]))
    plt.ylim(float(low[1]), float(high[1]))
    plt.title("GBS final xT samples")
    plt.show()

    return fwd_state, bwd_state, hist


In [18]:
# Choose one target:
low = jnp.array([0.0, 0.0])
high = jnp.array([1.0, 1.0])

logprob_fn = target1_logprob
# logprob_fn = target2_logprob
logprob_fn = target3_logprob
T= 10000
snap_iters = [0, 1, 2, 5, 10, 15, 24] + list(range(30, T, 10))
fwd_state, bwd_state, hist = run_gbs_toy(
    logprob_fn=logprob_fn,
    low=low,
    high=high,
    dim=dim,
    T=T,
    batch_size=512,
    num_steps=25,
    gif_path="GBS_training.gif",
    snap_iters=snap_iters,
    seed=0,
    loss_mode="tr_lv"
)


  0%|          | 0/10000 [00:00<?, ?it/s]


TypeError: max got incompatible shapes for broadcasting: (2,), (10,).